In [1]:
# Parameters
base_path = "C:\\TCC_USP"
run_id = "20251126_163824"


In [2]:
# 1. Setup de caminhos locais
import os
import subprocess
from datetime import datetime
from pathlib import Path

from src.io import paths
from src.config import loader as cfg

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "03_tfidf_models"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("BASE_PATH:", BASE_PATH)
print("PROC_PATH:", PROC_PATH)


BASE_PATH: C:\TCC_USP
PROC_PATH: C:\TCC_USP\data_processed


In [3]:
# 2. Carregar dados processados
import pandas as pd, numpy as np

df_ibov = pd.read_csv(os.path.join(PROC_PATH, "ibovespa_clean.csv"))
df_news = pd.read_csv(os.path.join(PROC_PATH, "noticias_clean.csv"))

# corrigir nomes das colunas (date -> data)
if "date" in df_ibov.columns: df_ibov.rename(columns={"date":"data"}, inplace=True)
if "date" in df_news.columns: df_news.rename(columns={"date":"data"}, inplace=True)
df_ibov["data"] = pd.to_datetime(df_ibov["data"])
df_news["data"] = pd.to_datetime(df_news["data"])

print("IBOV:", df_ibov.shape, "NEWS:", df_news.shape)
display(df_news.head())

IBOV: (1960, 9) NEWS: (10, 6)


,data,fonte,titulo,texto,sentimento,clean_text
0,2024-01-02,exemplo,Titulo 1,Ibovespa sobe com otimismo no mercado internac...,1,ibovespa sobe otimismo mercado internacional
1,2024-01-03,exemplo,Titulo 2,Bolsa cai após dados fracos da economia chinesa.,0,bolsa cai após dados fracos economia chinesa
2,2024-01-04,exemplo,Titulo 3,Petróleo em alta puxa ações da Petrobras para ...,1,petróleo alta puxa ações petrobras cima
3,2024-01-05,exemplo,Titulo 4,Queda do dólar beneficia empresas exportadoras.,1,queda dólar beneficia empresas exportadoras
4,2024-01-08,exemplo,Titulo 5,Setor bancário recua com aversão ao risco.,0,setor bancário recua aversão risco


In [4]:
# 3. TF-IDF para notÃ­cias
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=500, ngram_range=(1,2))
X_tfidf = vectorizer.fit_transform(df_news["clean_text"].fillna(""))

tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=vectorizer.get_feature_names_out())
tfidf_df["data"] = df_news["data"].values

# Agregar por dia (mÃ©dia dos vetores das notÃ­cias do dia)
daily_tfidf = tfidf_df.groupby("data").mean().reset_index()
print("Daily TF-IDF shape:", daily_tfidf.shape)

Daily TF-IDF shape: (10, 94)


In [5]:
# 4. Merge com IBOV e criar target
df = pd.merge(df_ibov.sort_values("data"), daily_tfidf, on="data", how="left").fillna(0)
df["ret1"] = df["close"].pct_change().shift(-1)           # retorno do dia seguinte
df = df.dropna().copy()
df["y"] = (df["ret1"] > 0).astype(int)

X = df.drop(columns=["data","close","ret1","y"]).values
y = df["y"].values

print("Final dataset:", X.shape, y.shape)

Final dataset: (1959, 100) (1959,)


In [6]:
# 5. Modelos: Logit, RandomForest, XGBoost
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

models = {
    "Logistic": LogisticRegression(max_iter=2000),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1,
                             subsample=0.8, colsample_bytree=0.8, random_state=42)
}

tscv = TimeSeriesSplit(n_splits=3)
results = {}

for name, clf in models.items():
    aucs, mdas = [], []
    for tr, te in tscv.split(X):
        clf.fit(X[tr], y[tr])
        proba = clf.predict_proba(X[te])[:,1]
        pred  = (proba > 0.5).astype(int)
        aucs.append(roc_auc_score(y[te], proba))
        mdas.append(accuracy_score(y[te], pred))
    results[name] = {"AUC": np.mean(aucs), "MDA": np.mean(mdas)}

print("Resultados mÃ©dios (TF-IDF):")
print(results)

Resultados mÃ©dios (TF-IDF):
{'Logistic': {'AUC': np.float64(0.5001121356801345), 'MDA': np.float64(0.5051124744376279)}, 'RandomForest': {'AUC': np.float64(0.5027840104423483), 'MDA': np.float64(0.5112474437627812)}, 'XGBoost': {'AUC': np.float64(0.5039149254888757), 'MDA': np.float64(0.4976141785957737)}}


In [7]:
# 6. Salvar resultados em JSON (para o dashboard)
import json, os

nb_name = "03_tfidf_models"

# results jÃ¡ estÃ¡ definido pelos modelos
results_dict = {k: {"AUC": float(v["AUC"]), "MDA": float(v["MDA"])} for k, v in results.items()}

out_file = os.path.join(PROC_PATH, f"results_{nb_name}.json")
with open(out_file, "w") as f:
    json.dump(results_dict, f, indent=4)

print(f"âœ… Resultados salvos em {out_file}")

âœ… Resultados salvos em C:\TCC_USP\data_processed\results_03_tfidf_models.json
